In [1]:
import os
import numpy as np
import pandas as pd
from PIL import Image

from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

from tensorflow import keras
import tensorflow as tf

from tensorflow.python.ops.numpy_ops import np_config
np_config.enable_numpy_behavior()

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:1024"
os.environ["CUDA_VISIBLE_DEVICES"]="0"

2026-09-15 04:59:52.332815: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
from pathlib import Path

def _find_repo_root(start: Path) -> Path:
    """Walk upward from `start` to find the directory containing this repo's data/
    folder, so this notebook works regardless of the kernel's working directory
    (VS Code defaults to the notebook's own folder; classic Jupyter defaults to
    wherever you launched it from)."""
    for candidate in [start.resolve()] + list(start.resolve().parents):
        if (candidate / "data" / "labels" / "mb24").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate the repo root (a data/labels/mb24 directory) above "
        f"{start}. Make sure data.tar.gz has been extracted per the main README."
    )

REPO_ROOT = _find_repo_root(Path.cwd())
DATA_DIR = REPO_ROOT / "data"
print(f"Using DATA_DIR = {DATA_DIR}")


Using DATA_DIR = /workspace/LFreeDA/data


## Useful functions 

In [3]:
import warnings
from concurrent.futures import ThreadPoolExecutor

# Some malware-derived and benign images legitimately exceed Pillow's default
# decompression-bomb pixel-count threshold. Disable the check globally (once,
# for both loaders below) instead of per-call, and silence the corresponding
# warning so it doesn't spam stdout during normal loading.
Image.MAX_IMAGE_PIXELS = None
warnings.filterwarnings("ignore", category=Image.DecompressionBombWarning)

def change_attack_label(x):
    label = [1.]
    return label

def _load_one_malware_image(args):
    filename, image_path = args
    hash_id = filename.split(".")[0]
    f = os.path.join(image_path, filename)
    image = Image.open(f).convert('RGB')
    image = image.resize((56, 56), Image.LANCZOS)
    image = np.array(image, dtype=int)
    return hash_id, image, image_path + "/" + filename

def load_image_malware(image_path, label_path, max_workers=None):
    labels = pd.read_csv(label_path, header=0)
    # O(1) label lookup instead of re-scanning the whole dataframe per image
    label_map = dict(zip(labels["malware SHA-256"], labels["Label"]))

    filenames = [
        filename for filename in os.listdir(image_path)
        if filename.endswith(".png") and filename.split(".")[0] in label_map
    ]

    # Image decode/resize is I/O- and PIL-bound (PIL releases the GIL for
    # most of this work), so a thread pool parallelizes it well without the
    # process-pool overhead of pickling images back to the main process.
    max_workers = max_workers or os.cpu_count()
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        results = list(executor.map(
            lambda fn: _load_one_malware_image((fn, image_path)), filenames
        ))

    x = [image for _, image, _ in results]
    y = [[label_map[hash_id]] for hash_id, _, _ in results]
    paths = [p for _, _, p in results]

    x = np.asarray(x)
    y = np.asarray(y)
    x = x.astype('float32') / 255.
    paths = np.array(paths, dtype=object)

    return x, y, paths

def _load_one_normal_image(args):
    filename, directory_path, image_size_limit = args
    file_path = os.path.join(directory_path, filename)
    try:
        with Image.open(file_path) as img:
            # Check if the image size is within the allowed limit
            if img.width * img.height <= image_size_limit:
                img = img.convert('RGB')
                img = img.resize((56, 56), Image.LANCZOS)
                return np.array(img, dtype=int), directory_path + "/" + filename
            else:
                # Silently skip oversized images (still skipped, just not logged).
                return None
    except (Image.DecompressionBombError, OSError) as e:
        print(f"Error loading image {filename}: {e}")
        return None

def load_image_normal(directory_path, max_workers=None):
    image_size_limit = 178956970  # Maximum allowed pixels per image

    filenames = [
        filename for filename in os.listdir(directory_path)
        if filename.endswith(".jpg") or filename.endswith(".png") or filename.endswith(".jpeg")
    ]

    max_workers = max_workers or os.cpu_count()
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        results = list(executor.map(
            lambda fn: _load_one_normal_image((fn, directory_path, image_size_limit)), filenames
        ))

    results = [r for r in results if r is not None]
    image_list = [img for img, _ in results]
    paths = [p for _, p in results]

    image_list = np.asarray(image_list)
    image_list = image_list.astype('float32') / 255.
    paths = np.array(paths, dtype=object)

    return image_list, paths


## Load malware data

### march

In [4]:
label_path = str(DATA_DIR / "labels/mb24/March/march_malware.csv")
img_path = str(DATA_DIR / "image_features/mb24/march/march")
malware_march_x, malware_march_y, malware_march_path=load_image_malware(img_path, label_path)
print(malware_march_x.shape)
print(malware_march_y.shape)

(1505, 56, 56, 3)
(1505, 1)


### april

In [5]:
label_path = str(DATA_DIR / "labels/mb24/April/april_malware.csv")
img_path = str(DATA_DIR / "image_features/mb24/april/april")
malware_april_x, malware_april_y,malware_april_path =load_image_malware(img_path, label_path)
print(malware_april_x.shape)
print(malware_april_y.shape)

(1080, 56, 56, 3)
(1080, 1)


### may

In [6]:
label_path = str(DATA_DIR / "labels/mb24/May/may_malware.csv")
img_path = str(DATA_DIR / "image_features/mb24/may/may")
malware_may_x, malware_may_y, malware_may_path =load_image_malware(img_path, label_path)
print(malware_may_x.shape)
print(malware_may_y.shape)

(1496, 56, 56, 3)
(1496, 1)


### july

In [7]:
label_path = str(DATA_DIR / "labels/mb24/July/july_malware.csv")
img_path = str(DATA_DIR / "image_features/mb24/july/july")
malware_july_x, malware_july_y, malware_july_path =load_image_malware(img_path, label_path)
print(malware_july_x.shape)
print(malware_july_y.shape)

(1618, 56, 56, 3)
(1618, 1)


### aug

In [8]:
label_path = str(DATA_DIR / "labels/mb24/Aug/aug_malware.csv")
img_path = str(DATA_DIR / "image_features/mb24/aug/aug")
malware_aug_x, malware_aug_y, malware_aug_path =load_image_malware(img_path, label_path)
print(malware_aug_x.shape)
print(malware_aug_y.shape)

(1613, 56, 56, 3)
(1613, 1)


## Load normal data source

In [9]:
img_path = str(DATA_DIR / "image_features/benign_source/dataset1")
source_normal_x_1, source_normal_path_1 = load_image_normal(img_path)
print(source_normal_x_1.shape)

(3000, 56, 56, 3)


In [10]:
source_normal_y_1 = np.zeros((source_normal_x_1.shape[0],1))

In [11]:
img_path = str(DATA_DIR / "image_features/benign_source/dataset2")
source_normal_x_2, source_normal_path_2 = load_image_normal(img_path)
print(source_normal_x_2.shape)

(2609, 56, 56, 3)


In [12]:
source_normal_y_2 = np.zeros((source_normal_x_2.shape[0],1))

In [13]:
img_path = str(DATA_DIR / "image_features/benign_source/dataset3")
source_normal_x_3, source_normal_path_3 = load_image_normal(img_path)
print(source_normal_x_3.shape)

(1526, 56, 56, 3)


In [14]:
source_normal_y_3 = np.zeros((source_normal_x_3.shape[0],1))

In [15]:
img_path = str(DATA_DIR / "image_features/benign_source/dataset4")
source_normal_x_4, source_normal_path_4 = load_image_normal(img_path)
print(source_normal_x_4.shape)

(1073, 56, 56, 3)


In [16]:
source_normal_y_4 = np.zeros((source_normal_x_4.shape[0],1))

### merge

In [17]:
source_normal_x = np.concatenate((source_normal_x_1, source_normal_x_2, source_normal_x_3, source_normal_x_4), axis = 0)
source_normal_y = np.concatenate((source_normal_y_1, source_normal_y_2, source_normal_y_3, source_normal_y_4), axis = 0)
source_normal_path = np.concatenate((source_normal_path_1, source_normal_path_2, source_normal_path_3, source_normal_path_4), axis = 0)

## Load normal data target

In [18]:
img_path = str(DATA_DIR / "image_features/benign_target/dataset1")
target_normal_x_1, target_normal_path_1 = load_image_normal(img_path)
print(target_normal_x_1.shape)

(1000, 56, 56, 3)


In [19]:
target_normal_y_1 = np.zeros((target_normal_x_1.shape[0],1))

In [20]:
img_path = str(DATA_DIR / "image_features/benign_target/dataset2")
target_normal_x_2, target_normal_path_2 = load_image_normal(img_path)
print(target_normal_x_2.shape)

(5740, 56, 56, 3)


In [21]:
target_normal_y_2 = np.zeros((target_normal_x_2.shape[0],1))

In [22]:
img_path = str(DATA_DIR / "image_features/benign_target/dataset3")
target_normal_x_3, target_normal_path_3 = load_image_normal(img_path)
print(target_normal_x_3.shape)

(1907, 56, 56, 3)


In [23]:
target_normal_y_3 = np.zeros((target_normal_x_3.shape[0],1))

In [24]:
img_path = str(DATA_DIR / "image_features/benign_target/dataset4")
target_normal_x_4, target_normal_path_4 = load_image_normal(img_path)
print(target_normal_x_4.shape)

(503, 56, 56, 3)


In [25]:
target_normal_y_4 = np.zeros((target_normal_x_4.shape[0],1))

### merge

In [26]:
target_normal_x = np.concatenate((target_normal_x_1, target_normal_x_2, target_normal_x_3, target_normal_x_4), axis = 0)
target_normal_y = np.concatenate((target_normal_y_1, target_normal_y_2, target_normal_y_3, target_normal_y_4), axis = 0)
target_normal_path = np.concatenate((target_normal_path_1, target_normal_path_2, target_normal_path_3, target_normal_path_4), axis = 0)

## Aug 

In [27]:
source_malware_x = np.concatenate((malware_march_x, malware_april_x, malware_may_x), axis = 0)
source_malware_y = np.concatenate((malware_march_y, malware_april_y, malware_may_y), axis = 0)
source_malware_path = np.concatenate((malware_march_path, malware_april_path, malware_may_path), axis = 0)

target_malware_train_x = malware_july_x
target_malware_train_y = malware_july_y
target_malware_train_path = malware_july_path

target_malware_test_x = malware_aug_x
target_malware_test_y = malware_aug_y
target_malware_test_path = malware_aug_path

target_normal_train_x, target_normal_test_x, \
target_normal_train_y, target_normal_test_y, \
target_normal_train_path, target_normal_test_path = train_test_split(target_normal_x, target_normal_y, target_normal_path, test_size=0.5, random_state=42)



print("Malware data ...")
print("Target train: {}".format(target_malware_train_x.shape))
print("Target train; {}".format(target_malware_train_y.shape))
print("Target test: {}".format(target_malware_test_x.shape))
print("Target test; {}".format(target_malware_test_y.shape))
print("Source {}".format(source_malware_x.shape))
print("Source {}".format(source_malware_y.shape))
print("=============================================")

print("Normal data ...")
print("Target orgin : {}".format(target_normal_x.shape))
print("Target orign : {}".format(target_normal_y.shape))
print("Source {}".format(source_normal_x.shape))
print("Source {}".format(source_normal_y.shape))

print("=============================================")

print("Normal data target train...")
print("Target orgin : {}".format(target_normal_train_x.shape))
print("Target orign : {}".format(target_normal_train_y.shape))

print("=============================================")


print("Normal data target test...")
print("Target orgin : {}".format(target_normal_test_x.shape))
print("Target orign : {}".format(target_normal_test_y.shape))

print("=============================================")




source_malware_y = np.apply_along_axis(change_attack_label, 1, source_malware_y)
target_malware_train_y = np.apply_along_axis(change_attack_label, 1, target_malware_train_y)
target_malware_test_y = np.apply_along_axis(change_attack_label, 1, target_malware_test_y)


source_x = np.concatenate((source_malware_x, source_normal_x), axis = 0)
source_y = np.concatenate((source_malware_y, source_normal_y), axis = 0)
source_path = np.concatenate((source_malware_path, source_normal_path), axis = 0)



target_x_train = np.concatenate((target_malware_train_x, target_normal_train_x), axis = 0)
target_y_train = np.concatenate((target_malware_train_y, target_normal_train_y), axis = 0)
target_path_train = np.concatenate((target_malware_train_path, target_normal_train_path), axis = 0)


target_x_test = np.concatenate((target_malware_test_x, target_normal_test_x), axis = 0)
target_y_test = np.concatenate((target_malware_test_y, target_normal_test_y), axis = 0)
target_path_test = np.concatenate((target_malware_test_path, target_normal_test_path), axis = 0)


#one-hot encode labels

source_y = tf.keras.utils.to_categorical(source_y, num_classes = 2)
target_y_train  = tf.keras.utils.to_categorical(target_y_train, num_classes = 2)
target_y_test = tf.keras.utils.to_categorical(target_y_test , num_classes = 2)



source_x_train, source_x_test, \
source_y_train, source_y_test, \
source_path_train, source_path_test= train_test_split(source_x, source_y, source_path, test_size=0.25, random_state=42)


print("train test data ...")
print("Target train: {}".format(target_x_train.shape))
print("Target train: {}".format(target_y_train.shape))
print("Target test: {}".format(target_x_test.shape))
print("Target test: {}".format(target_y_test.shape))
print("Source train: {}".format(source_x_train.shape))
print("Source train: {}".format(source_y_train.shape))
print("Source test: {}".format(source_x_test.shape))
print("Source test: {}".format(source_y_test.shape))

Malware data ...
Target train: (1618, 56, 56, 3)
Target train; (1618, 1)
Target test: (1613, 56, 56, 3)
Target test; (1613, 1)
Source (4081, 56, 56, 3)
Source (4081, 1)
Normal data ...
Target orgin : (9150, 56, 56, 3)
Target orign : (9150, 1)
Source (8208, 56, 56, 3)
Source (8208, 1)
Normal data target train...
Target orgin : (4575, 56, 56, 3)
Target orign : (4575, 1)
Normal data target test...
Target orgin : (4575, 56, 56, 3)
Target orign : (4575, 1)


train test data ...
Target train: (6193, 56, 56, 3)
Target train: (6193, 2)
Target test: (6188, 56, 56, 3)
Target test: (6188, 2)
Source train: (9216, 56, 56, 3)
Source train: (9216, 2)
Source test: (3073, 56, 56, 3)
Source test: (3073, 2)


### Load MaxDIRep

In [28]:
# …later, or in a new script/session…
generator = keras.models.load_model(str(DATA_DIR / "stepI_trained_models/mb24/aug/generator"))
classifier = keras.models.load_model(str(DATA_DIR / "stepI_trained_models/mb24/aug/classifier"))


y_target_class_pred = classifier.predict(generator(target_x_train)).argmax(1)

# 1. Original noisy‐label accuracy on training data
y_noisy = y_target_class_pred  
y_true_all = target_y_train.argmax(axis=1)
acc_orig = accuracy_score(y_true_all, y_target_class_pred)
print(f"Original noisy accuracy on target train: {acc_orig:.2%} "
      f"(on {len(y_true_all)} samples)")

# target_y_test = target_y_test.argmax(axis=1)
# source_y = source_y.argmax(axis=1)

2026-09-15 05:00:50.545600: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-09-15 05:00:50.938122: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1532] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 21993 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:01:00.0, compute capability: 8.9


2026-09-15 05:00:51.888922: I tensorflow/stream_executor/cuda/cuda_dnn.cc:384] Loaded cuDNN version 8100


  1/194 [..............................] - ETA: 8s

104/194 [===============>..............] - ETA: 0s

194/194 [==============================] - 0s 481us/step


Original noisy accuracy on target train: 81.16% (on 6193 samples)


2026-09-15 05:00:52.967330: I tensorflow/stream_executor/cuda/cuda_blas.cc:1786] TensorFloat-32 will be used for the matrix multiplication. This will only be logged once.


In [29]:
# target_x_train:  (N, …)
Z_maps = generator.predict(target_x_train)        # shape = (N, 12, 12, 64)
N = Z_maps.shape[0]

Z = Z_maps.reshape(N, -1)    
 

  1/194 [..............................] - ETA: 6s

 91/194 [=============>................] - ETA: 0s

184/194 [===========================>..] - ETA: 0s

194/194 [==============================] - 0s 577us/step


### local outlier factor

In [30]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import accuracy_score

# 1. PCA → 50 dims (speed + regularization)
pca = PCA(n_components=50, svd_solver="randomized", random_state=0)
Z_reduced = pca.fit_transform(Z)  # (N, 50)

# 2. Softmax probabilities + confidence
probs = classifier.predict(Z)  # shape = (N, num_classes)
conf = np.max(probs, axis=1)           # best-class probability

# 3. Build masks: LOF-only, confidence-only, and combined
keep_lof   = np.zeros(N, dtype=bool)
keep_conf  = np.zeros(N, dtype=bool)
keep_combo = np.zeros(N, dtype=bool)
lof_contam = 0.2 # fraction of outliers per class
conf_thresh = 0.95  # confidence cutoff

#y_noisy = y_target_class_pred
# 4. Per-class LOF filtering + confidence gating
for c in np.unique(y_noisy):
    idx = np.where(y_noisy == c)[0]
    Zc = Z_reduced[idx]

    # fit LOF on class-c embeddings
    lof = LocalOutlierFactor(n_neighbors=50, contamination=lof_contam)
    preds = lof.fit_predict(Zc)   # +1=inlier, -1=outlier
    inliers = preds == 1

    # update LOF-only mask
    keep_lof[idx[inliers]] = True

    # update confidence-only mask
    conf_mask = conf[idx] >= conf_thresh
    keep_conf[idx[conf_mask]] = True

    # update combined mask
    keep_combo[idx[inliers & conf_mask]] = True

# 5. Slice out subsets
# y_true_all  = target_y_train.argmax(axis=1)

y_true_lof   = y_true_all[keep_lof]
y_pred_lof   = y_noisy[keep_lof]


y_true_conf  = y_true_all[keep_conf]
y_pred_conf  = y_noisy[keep_conf]


y_true_combo = y_true_all[keep_combo]
y_pred_combo = y_noisy[keep_combo]

# 6. Compute & print accuracies
acc_orig  = accuracy_score(y_true_all, y_noisy)
acc_lof   = accuracy_score(y_true_lof, y_pred_lof)
acc_conf  = accuracy_score(y_true_conf, y_pred_conf)
acc_combo = accuracy_score(y_true_combo, y_pred_combo)

print(f"Original accuracy             : {acc_orig:.2%} on {N} samples")
print(f"After LOF-only                : {acc_lof:.2%} "
      f"({keep_lof.sum()}/{N} ≈ {keep_lof.mean():.1%} retained)")
print(f"After confidence-only         : {acc_conf:.2%} "
      f"({keep_conf.sum()}/{N} ≈ {keep_conf.mean():.1%} retained)")
print(f"After LOF + confidence        : {acc_combo:.2%} "
      f"({keep_combo.sum()}/{N} ≈ {keep_combo.mean():.1%} retained)\n")

# 7. Per-class counts for LOF-only
print("Kept per class (LOF-only):")
for c in np.unique(y_noisy):
    total = np.sum(y_noisy == c)
    kept  = np.sum((y_noisy == c) & keep_lof)
    print(f"  class {c}: kept {kept}/{total} = {kept/total:.1%}")

# 8. Per-class counts for confidence-only
print("\nKept per class (confidence-only):")
for c in np.unique(y_noisy):
    total = np.sum(y_noisy == c)
    kept  = np.sum((y_noisy == c) & keep_conf)
    print(f"  class {c}: kept {kept}/{total} = {kept/total:.1%}")

# 9. Per-class counts for LOF + confidence
print("\nKept per class (LOF + confidence):")
for c in np.unique(y_noisy):
    total = np.sum(y_noisy == c)
    kept  = np.sum((y_noisy == c) & keep_combo)
    print(f"  class {c}: kept {kept}/{total} = {kept/total:.1%}")
    

  1/194 [..............................] - ETA: 4s

102/194 [==============>...............] - ETA: 0s

194/194 [==============================] - 0s 492us/step


Original accuracy             : 81.16% on 6193 samples
After LOF-only                : 82.72% (4954/6193 ≈ 80.0% retained)
After confidence-only         : 87.51% (4773/6193 ≈ 77.1% retained)
After LOF + confidence        : 89.17% (3831/6193 ≈ 61.9% retained)

Kept per class (LOF-only):
  class 0: kept 3435/4294 = 80.0%
  class 1: kept 1519/1899 = 80.0%

Kept per class (confidence-only):
  class 0: kept 3683/4294 = 85.8%
  class 1: kept 1090/1899 = 57.4%

Kept per class (LOF + confidence):
  class 0: kept 2996/4294 = 69.8%
  class 1: kept 835/1899 = 44.0%


In [31]:
target_x_train_filtered = target_x_train[keep_combo]
target_pred_train_filtered = y_pred_combo
target_path_train_filtered  = target_path_train[keep_combo]

### GMM

In [32]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.metrics import accuracy_score

# 1. PCA → 50 dims (speed + regularization)
pca = PCA(n_components=50, svd_solver="randomized", random_state=0)
Z_reduced = pca.fit_transform(Z)                 # (N, 50)

# 2. Softmax probabilities + confidence
probs = classifier.predict(Z)            # shape = (N, num_classes)
conf  = np.max(probs, axis=1)                    # best-class probability

# 3. Build masks: GMM-only, confidence-only, and combined
keep_gmm   = np.zeros(N, dtype=bool)
keep_conf  = np.zeros(N, dtype=bool)
keep_combo = np.zeros(N, dtype=bool)
gmm_contam = 0.20    # fraction of outliers per class
conf_thresh = 0.95   # confidence cutoff

# y_noisy = y_target_class_pred

# 4. Per-class GMM filtering + confidence gating
for c in np.unique(y_noisy):
    idx = np.where(y_noisy == c)[0]
    Zc  = Z_reduced[idx]                        # embeddings for class c

    # fit a single-component GMM
    gmm = GaussianMixture(n_components=1,
                          covariance_type='full',
                          reg_covar=1e-6,
                          random_state=0)
    gmm.fit(Zc)

    # compute log-likelihoods
    log_probs = gmm.score_samples(Zc)           # shape = (n_c,)

    # GMM-only mask (inliers above quantile)
    thresh = np.percentile(log_probs, gmm_contam * 100)
    inliers = log_probs > thresh
    keep_gmm[idx[inliers]] = True

    # confidence-only mask
    conf_mask = conf[idx] >= conf_thresh
    keep_conf[idx[conf_mask]] = True

    # combined mask
    keep_combo[idx[inliers & conf_mask]] = True

# 5. Slice out subsets
y_true_all   = target_y_train.argmax(axis=1)
y_true_gmm   = y_true_all[keep_gmm]
y_pred_gmm   = y_noisy[keep_gmm]

y_true_conf  = y_true_all[keep_conf]
y_pred_conf  = y_noisy[keep_conf]

y_true_combo = y_true_all[keep_combo]
y_pred_combo = y_noisy[keep_combo]

# 6. Compute & print accuracies
acc_orig = accuracy_score(y_true_all, y_noisy)
acc_gmm  = accuracy_score(y_true_gmm, y_pred_gmm)
acc_conf = accuracy_score(y_true_conf, y_pred_conf)
acc_combo= accuracy_score(y_true_combo, y_pred_combo)

print(f"Original accuracy               : {acc_orig:.2%} on {N} samples")
print(f"After GMM-only                  : {acc_gmm:.2%} "
      f"({keep_gmm.sum()}/{N} ≈ {keep_gmm.mean():.1%} retained)")
print(f"After confidence-only           : {acc_conf:.2%} "
      f"({keep_conf.sum()}/{N} ≈ {keep_conf.mean():.1%} retained)")
print(f"After GMM + confidence          : {acc_combo:.2%} "
      f"({keep_combo.sum()}/{N} ≈ {keep_combo.mean():.1%} retained)\n")

# 7. Per-class counts for GMM-only
print("Kept per class (GMM-only):")
for c in np.unique(y_noisy):
    total = np.sum(y_noisy == c)
    kept  = np.sum((y_noisy == c) & keep_gmm)
    print(f"  class {c}: kept {kept}/{total} = {kept/total:.1%}")

# 8. Per-class counts for confidence-only
print("\nKept per class (confidence-only):")
for c in np.unique(y_noisy):
    total = np.sum(y_noisy == c)
    kept  = np.sum((y_noisy == c) & keep_conf)
    print(f"  class {c}: kept {kept}/{total} = {kept/total:.1%}")

# 9. Per-class counts for GMM + confidence
print("\nKept per class (GMM + confidence):")
for c in np.unique(y_noisy):
    total = np.sum(y_noisy == c)
    kept  = np.sum((y_noisy == c) & keep_combo)
    print(f"  class {c}: kept {kept}/{total} = {kept/total:.1%}")


  1/194 [..............................] - ETA: 2s

 97/194 [==============>...............] - ETA: 0s

194/194 [==============================] - 0s 511us/step


Original accuracy               : 81.16% on 6193 samples
After GMM-only                  : 81.31% (4954/6193 ≈ 80.0% retained)
After confidence-only           : 87.51% (4773/6193 ≈ 77.1% retained)
After GMM + confidence          : 88.56% (3654/6193 ≈ 59.0% retained)

Kept per class (GMM-only):
  class 0: kept 3435/4294 = 80.0%
  class 1: kept 1519/1899 = 80.0%

Kept per class (confidence-only):
  class 0: kept 3683/4294 = 85.8%
  class 1: kept 1090/1899 = 57.4%

Kept per class (GMM + confidence):
  class 0: kept 2855/4294 = 66.5%
  class 1: kept 799/1899 = 42.1%


### One class svm 

In [33]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.svm import OneClassSVM
from sklearn.metrics import accuracy_score

# 1. PCA → 50 dims (speed + regularization)
pca = PCA(n_components=50, svd_solver="randomized", random_state=0)
Z_reduced = pca.fit_transform(Z)                 # (N, 50)

# 2. Softmax probabilities + confidence
probs = classifier.predict(Z)            # shape = (N, num_classes)
conf  = np.max(probs, axis=1)                    # best-class probability

# 3. Build masks: One-Class SVM only, confidence-only, and combined
keep_ocsvm = np.zeros(N, dtype=bool)
keep_conf  = np.zeros(N, dtype=bool)
keep_combo = np.zeros(N, dtype=bool)
svm_nu     = 0.2      # fraction of outliers per class
aic_conf_thresh = 0.95   # confidence cutoff

# 4. Per-class One-Class SVM filtering + confidence gating
y_noisy = y_target_class_pred
for c in np.unique(y_noisy):
    idx = np.where(y_noisy == c)[0]
    Zc  = Z_reduced[idx]                        # embeddings for class c

    # fit One-Class SVM
    ocsvm = OneClassSVM(nu=svm_nu, kernel='rbf', gamma='auto')
    ocsvm.fit(Zc)
    preds = ocsvm.predict(Zc)                   # +1=inlier, -1=outlier
    inliers = preds == 1

    # update One-Class SVM mask
    keep_ocsvm[idx[inliers]] = True

    # update confidence mask
    conf_mask = conf[idx] >= aic_conf_thresh
    keep_conf[idx[conf_mask]] = True

    # update combined mask
    keep_combo[idx[inliers & conf_mask]] = True

# 5. Slice out subsets
y_true_all     = target_y_train.argmax(axis=1)
y_true_ocsvm   = y_true_all[keep_ocsvm]
y_pred_ocsvm   = y_noisy[keep_ocsvm]

y_true_conf    = y_true_all[keep_conf]
y_pred_conf    = y_noisy[keep_conf]

y_true_combo   = y_true_all[keep_combo]
y_pred_combo   = y_noisy[keep_combo]

# 6. Compute & print accuracies
acc_orig   = accuracy_score(y_true_all, y_noisy)
acc_ocsvm  = accuracy_score(y_true_ocsvm, y_pred_ocsvm)
acc_conf   = accuracy_score(y_true_conf, y_pred_conf)
acc_combo  = accuracy_score(y_true_combo, y_pred_combo)

print(f"Original accuracy                : {acc_orig:.2%} on {N} samples")
print(f"After One-Class SVM only         : {acc_ocsvm:.2%} "
      f"({keep_ocsvm.sum()}/{N} ≈ {keep_ocsvm.mean():.1%} retained)")
print(f"After confidence-only            : {acc_conf:.2%} "
      f"({keep_conf.sum()}/{N} ≈ {keep_conf.mean():.1%} retained)")
print(f"After One-Class SVM + confidence : {acc_combo:.2%} "
      f"({keep_combo.sum()}/{N} ≈ {keep_combo.mean():.1%} retained)\n")

# 7. Per-class retention counts
print("Kept per class (One-Class SVM only):")
for c in np.unique(y_noisy):
    total = np.sum(y_noisy == c)
    kept  = np.sum((y_noisy == c) & keep_ocsvm)
    print(f"  class {c}: kept {kept}/{total} = {kept/total:.1%}")

print("\nKept per class (confidence-only):")
for c in np.unique(y_noisy):
    total = np.sum(y_noisy == c)
    kept  = np.sum((y_noisy == c) & keep_conf)
    print(f"  class {c}: kept {kept}/{total} = {kept/total:.1%}")

print("\nKept per class (One-Class SVM + confidence):")
for c in np.unique(y_noisy):
    total = np.sum(y_noisy == c)
    kept  = np.sum((y_noisy == c) & keep_combo)
    print(f"  class {c}: kept {kept}/{total} = {kept/total:.1%}")


  1/194 [..............................] - ETA: 4s

 95/194 [=============>................] - ETA: 0s

192/194 [============================>.] - ETA: 0s

194/194 [==============================] - 0s 530us/step


Original accuracy                : 81.16% on 6193 samples
After One-Class SVM only         : 81.11% (4961/6193 ≈ 80.1% retained)
After confidence-only            : 87.51% (4773/6193 ≈ 77.1% retained)
After One-Class SVM + confidence : 88.59% (3645/6193 ≈ 58.9% retained)

Kept per class (One-Class SVM only):
  class 0: kept 3443/4294 = 80.2%
  class 1: kept 1518/1899 = 79.9%

Kept per class (confidence-only):
  class 0: kept 3683/4294 = 85.8%
  class 1: kept 1090/1899 = 57.4%

Kept per class (One-Class SVM + confidence):
  class 0: kept 2858/4294 = 66.6%
  class 1: kept 787/1899 = 41.4%


### Mahalanobis-distance

In [34]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.covariance import EmpiricalCovariance
from scipy.stats import chi2
from sklearn.metrics import accuracy_score


# 1. PCA → 50 dims (speed + regularization)
pca = PCA(n_components=50, svd_solver="randomized", random_state=0)
Z_reduced = pca.fit_transform(Z)  # (N, 50)

# 2. Softmax probabilities + confidence
probs = classifier.predict(Z)  # shape = (N, num_classes)
conf = np.max(probs, axis=1)    # best-class probability

# 3. Build masks: Mahalanobis-only, confidence-only, and Mahalanobis+confidence
keep_maha = np.zeros(N, dtype=bool)
keep_conf = np.zeros(N, dtype=bool)
keep_combo = np.zeros(N, dtype=bool)
maha_alpha = 0.8    # χ² percentile threshold
conf_thresh = 0.95  # confidence cutoff

# 4. Per-class filtering
# y_noisy = y_target_class_pred
for c in np.unique(y_noisy):
    idx = np.where(y_noisy == c)[0]
    Zc = Z_reduced[idx]

    # Mahalanobis distances
    cov_est = EmpiricalCovariance().fit(Zc)
    m2 = cov_est.mahalanobis(Zc)
    maha_thresh = chi2.ppf(maha_alpha, df=Zc.shape[1])
    maha_mask = m2 < maha_thresh

    # Confidence mask
    conf_mask = conf[idx] >= conf_thresh

    # Update masks
    keep_maha[idx[maha_mask]] = True
    keep_conf[idx[conf_mask]] = True
    keep_combo[idx[maha_mask & conf_mask]] = True

# 5. Slice subsets
y_true_all = target_y_train.argmax(axis=1)
y_true_maha = y_true_all[keep_maha]
y_pred_maha = y_noisy[keep_maha]

y_true_conf = y_true_all[keep_conf]
y_pred_conf = y_noisy[keep_conf]

y_true_combo = y_true_all[keep_combo]
y_pred_combo = y_noisy[keep_combo]

# 6. Compute & print accuracies
acc_orig = accuracy_score(y_true_all, y_noisy)
acc_maha = accuracy_score(y_true_maha, y_pred_maha)
acc_conf = accuracy_score(y_true_conf, y_pred_conf)
acc_combo = accuracy_score(y_true_combo, y_pred_combo)

print(f"Original accuracy                : {acc_orig:.2%} on {N} samples")
print(f"After Mahalanobis-only           : {acc_maha:.2%} "
      f"({keep_maha.sum()}/{N} ≈ {keep_maha.mean():.1%} retained)")
print(f"After confidence-only            : {acc_conf:.2%} "
      f"({keep_conf.sum()}/{N} ≈ {keep_conf.mean():.1%} retained)")
print(f"After Mahalanobis + confidence   : {acc_combo:.2%} "
      f"({keep_combo.sum()}/{N} ≈ {keep_combo.mean():.1%} retained)")

# 7. Per-class counts for each filter
print("Kept per class (Mahalanobis-only):")
for c in np.unique(y_noisy):
    total = np.sum(y_noisy == c)
    kept = np.sum((y_noisy == c) & keep_maha)
    print(f"  class {c}: kept {kept}/{total} = {kept/total:.1%}")

print("Kept per class (confidence-only):")
for c in np.unique(y_noisy):
    total = np.sum(y_noisy == c)
    kept = np.sum((y_noisy == c) & keep_conf)
    print(f"  class {c}: kept {kept}/{total} = {kept/total:.1%}")

print("Kept per class (Mahalanobis+confidence):")
for c in np.unique(y_noisy):
    total = np.sum(y_noisy == c)
    kept = np.sum((y_noisy == c) & keep_combo)
    print(f"  class {c}: kept {kept}/{total} = {kept/total:.1%}")


  1/194 [..............................] - ETA: 2s

 98/194 [==============>...............] - ETA: 0s

194/194 [==============================] - 0s 502us/step


Original accuracy                : 81.16% on 6193 samples
After Mahalanobis-only           : 81.41% (4793/6193 ≈ 77.4% retained)
After confidence-only            : 87.51% (4773/6193 ≈ 77.1% retained)
After Mahalanobis + confidence   : 88.74% (3517/6193 ≈ 56.8% retained)
Kept per class (Mahalanobis-only):
  class 0: kept 3329/4294 = 77.5%
  class 1: kept 1464/1899 = 77.1%
Kept per class (confidence-only):
  class 0: kept 3683/4294 = 85.8%
  class 1: kept 1090/1899 = 57.4%
Kept per class (Mahalanobis+confidence):
  class 0: kept 2752/4294 = 64.1%
  class 1: kept 765/1899 = 40.3%


### Isolation forest

In [35]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.ensemble     import IsolationForest
from sklearn.metrics      import accuracy_score

# 2. PCA → 50 dims (speed + regularization)
pca = PCA(n_components=50, svd_solver="randomized", random_state=0)
Z_reduced = pca.fit_transform(Z)                 # (N, 50)


# 4. Softmax probabilities + confidence
probs = classifier.predict(Z) # shape = (N, num_classes)
conf  = np.max(probs, axis=1)                    # best-class probability

# 5. Build two masks:
#    - IsolationForest only
#    - IsolationForest + confidence
keep_iforest = np.zeros(N, dtype=bool)
keep_conf   = np.zeros(N, dtype=bool)
keep_combo   = np.zeros(N, dtype=bool)

iso_contam   = 0.2  # fraction of outliers per class
conf_thresh  = 0.95  # confidence cutoff

for c in np.unique(y_noisy):
    idx = np.where(y_noisy == c)[0]
    Zc  = Z_reduced[idx]                        # embeddings for class c

    # fit IsolationForest on class-c embeddings
    iso = IsolationForest(contamination=iso_contam, random_state=0)
    iso.fit(Zc)
    preds = iso.predict(Zc)                     # +1 inlier, -1 outlier
    inliers = (preds == 1)

    # mark kept for IF only
    keep_iforest[idx[inliers]] = True

    # combine with high-confidence
    conf_mask = conf[idx] > conf_thresh
    keep_conf[idx[conf_mask]] = True
    
    keep_combo[idx[inliers & conf_mask]] = True

# 6. Slice out subsets
y_true_iforest  = y_true_all[keep_iforest]
y_pred_iforest  = y_noisy[keep_iforest]


y_true_conf  = y_true_all[keep_conf]
y_pred_conf  = y_noisy[keep_conf]

y_true_combo    = y_true_all[keep_combo]
y_pred_combo    = y_noisy[keep_combo]

# 7. Compute & print accuracies
acc_orig    = accuracy_score(y_true_all, y_noisy)
acc_iforest = accuracy_score(y_true_iforest, y_pred_iforest)
acc_conf = accuracy_score(y_true_conf, y_pred_conf)
acc_combo   = accuracy_score(y_true_combo, y_pred_combo)

print(f"Original accuracy             : {acc_orig:.2%} on {N} samples")
print(f"After IsolationForest only    : {acc_iforest:.2%} "
      f"({keep_iforest.sum()}/{N} ≈ {keep_iforest.mean():.1%} retained)")
print(f"After confidence filter     : {acc_conf:.2%} "
      f"({keep_conf.sum()}/{N} ≈ {keep_conf.mean():.1%} retained)")
print(f"After + confidence filter     : {acc_combo:.2%} "
      f"({keep_combo.sum()}/{N} ≈ {keep_combo.mean():.1%} retained)")

# Per‐class counts for IsolationForest‐only
print("\nKept per class (IsolationForest only):")
for c in np.unique(y_noisy):
    total = np.sum(y_noisy == c)
    kept  = np.sum(y_noisy[keep_iforest] == c)
    print(f"  class {c}: kept {kept}/{total} = {kept/total:.1%}")
    
    
# Per‐class counts for IsolationForest‐only
print("\nKept per class (confidence filter):")
for c in np.unique(y_noisy):
    total = np.sum(y_noisy == c)
    kept  = np.sum(y_noisy[keep_conf] == c)
    print(f"  class {c}: kept {kept}/{total} = {kept/total:.1%}")   

# Per‐class counts for IsolationForest + confidence
print("\nKept per class (IsolationForest + confidence):")
for c in np.unique(y_noisy):
    total = np.sum(y_noisy == c)
    kept  = np.sum(y_noisy[keep_combo] == c)
    print(f"  class {c}: kept {kept}/{total} = {kept/total:.1%}")


  1/194 [..............................] - ETA: 2s

101/194 [==============>...............] - ETA: 0s

194/194 [==============================] - 0s 493us/step


Original accuracy             : 81.16% on 6193 samples
After IsolationForest only    : 81.03% (4954/6193 ≈ 80.0% retained)
After confidence filter     : 87.51% (4773/6193 ≈ 77.1% retained)
After + confidence filter     : 88.33% (3649/6193 ≈ 58.9% retained)

Kept per class (IsolationForest only):
  class 0: kept 3435/4294 = 80.0%
  class 1: kept 1519/1899 = 80.0%

Kept per class (confidence filter):
  class 0: kept 3683/4294 = 85.8%
  class 1: kept 1090/1899 = 57.4%

Kept per class (IsolationForest + confidence):
  class 0: kept 2852/4294 = 66.4%
  class 1: kept 797/1899 = 42.0%


### Save the source and target 

In [36]:
# Replace these with your actual arrays:
# source_paths_train, target_paths_train_filtered, target_paths_test

# Uncomment below to save the constructed datasets. Saved under results/ (not data/)
# so a from-scratch run never overwrites the precomputed data/stepII_constructed_datasets/
# shipped with this repo. Nested under mb24/aug to mirror the real
# data/stepII_constructed_datasets/mb24/aug layout that StepIII's scripts expect.

# save_dir = str(REPO_ROOT / "results/stepII_constructed_datasets_scratch/mb24/aug")
# os.makedirs(save_dir, exist_ok=True)

# np.savez_compressed(f'{save_dir}/source_train.npz',
#                     source_path_train=source_path_train, source_y_train = source_y_train.argmax(axis=1))
# np.savez_compressed(f'{save_dir}/source_test.npz',
#                     source_path_test=source_path_test, source_y_test = source_y_test.argmax(axis=1))
# np.savez_compressed(f'{save_dir}/target_train_filtered.npz',
#                     target_path_train_filtered=target_path_train_filtered, target_pred_train_filtered=target_pred_train_filtered, target_true_train_filtered=y_true_combo)
# np.savez_compressed(f'{save_dir}/target_train.npz',
#                     target_path_train=target_path_train,target_y_train=target_y_train.argmax(axis=1))
# np.savez_compressed(f'{save_dir}/target_test.npz',
#                     target_path_test=target_path_test,target_y_test=target_y_test.argmax(axis=1))

# print("Saved Step II constructed datasets to {}".format(save_dir))